# Corrected candidate M0 baseline — Qwen3-1.7B

Candidate pilot-v1.1 corrects one independently confirmed book-title collision. It preserves the 30 F / 90 R / 60 N_rel units, splits, IDs and every fact used in the passing tiny pilot. No M1 adapter is loaded and no training occurs here.

This is a **derived baseline refresh**, not another full 2,520-response GPU run. First replay 64 answers from two unchanged original batches in both modes. Only if every saved output/token/score matches exactly, regenerate the complete affected eight-row batch in each mode (16 answers), then combine those 16 with 2,504 unchanged parent records, with per-row provenance. Same model pin, package versions, T4, prompts, seeds and batch shapes. Control mismatch stops merging and saves failure evidence; no automatic retries or relaxed checks.

Publish the prepared source/config/new candidate and verified parent evidence first. Use the existing Kaggle GPU kernel. Run all cells sequentially and download the final ZIP, including if controls fail. Full M1 and dataset acceptance remain pending manual corrected-row review. Thinking is the planned main research setting; both modes remain trained/evaluated under the successful mixed recipe.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='corrected-m0-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/corrected_m0.py').is_file(), 'Publish the prepared correction/refresh files before running.'
assert Path('data/candidates/pilot-v1.1/seed-42/checksums.sha256').is_file()
assert Path('experiments/runs/m0_qwen17_full_20261106/artifact_verification.json').is_file()
config_path = 'configs/evaluation/m0_qwen17_corrected_v1_1.json'


Cloning into '/kaggle/working/corrected-m0-source-32e3jsp3/repo'...


Repository commit: ed86cbaaae144a706dd2f2be3cee9d0cca13db10


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))'], check=True)


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
2.11.0+cu130 5.20.2 Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers; assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))'], returncode=0)

In [3]:
import json

# Independent complete-parent evidence check, no model inference.
subprocess.run([sys.executable, 'experiments/runs/m0_qwen17_full_20261106/verify_artifacts.py'], check=True)
plan_dir = Path(tempfile.mkdtemp(prefix='corrected-m0-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.corrected_m0', '--config', config_path, '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_new_predictions'] == 80 and plan['derived_predictions'] == 2520
assert plan['new_rows_in_derived_baseline'] == 16 and plan['reused_rows_in_derived_baseline'] == 2504
print('Verified both candidates/parent artifacts; planned GPU answers:', plan['planned_new_predictions'])
print('Affected batches:', plan['affected_batches'])


{
  "verified": true,
  "prediction_rows_rescored": 2520,
  "saved_token_records_checked": 2520,
  "offline_tokenizer_render_and_decode_replayed": false,
  "mode_counts": {
    "answer_only": {
      "answer_matches": 9,
      "non_year_matches": 0,
      "format_violations": 809,
      "token_limit_hits": 0,
      "missing_thinking_boundaries": 0,
      "reasoning_mentions": 0
    },
    "thinking": {
      "answer_matches": 1,
      "non_year_matches": 0,
      "format_violations": 752,
      "token_limit_hits": 16,
      "missing_thinking_boundaries": 15,
      "reasoning_mentions": 4
    }
  },
  "potential_answer_review_rows": 13,
  "protocol_issue_rows": 1562,
  "external_review_rows": 112,
  "limits": [
    "No model-weight/inference replay or proof of absent internal knowledge.",
    "Year matches/trace mentions are not evidence of the complete synthetic graph.",
    "Execution completion is not dataset acceptance; confirmed surface collision remains unresolved."
  ]
}
{
  "con

In [4]:
run_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    download_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    download_token = None
if download_token:
    run_env['HF_TOKEN'] = download_token
output_dir = Path(tempfile.mkdtemp(prefix='m0-qwen17-corrected-', dir=remote_root))
print('Refresh outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.corrected_m0', '--config', config_path, '--output-dir', str(output_dir), '--mode', 'audit'], env=run_env, check=True)
finally:
    run_env.pop('HF_TOKEN', None)


Refresh outputs: /kaggle/working/m0-qwen17-corrected-aqe8clfe


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 167.47it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


control_predictions.jsonl: answer_only/m1_dev offset=0, rows=16
control_predictions.jsonl: answer_only/n_rel_train offset=0, rows=16
control_predictions.jsonl: thinking/m1_dev offset=0, rows=16
control_predictions.jsonl: thinking/n_rel_train offset=0, rows=16
refreshed_predictions.jsonl: answer_only/m1_dev offset=832, rows=8
refreshed_predictions.jsonl: thinking/m1_dev offset=832, rows=8
{
  "execution_complete": true,
  "model_executed": true,
  "baseline_kind": "Derived: unchanged parent batches plus complete refreshed affected batches.",
  "replay_controls_passed": true,
  "new_predictions": 80,
  "reused_predictions": 2504,
  "training_updates": 0,
  "adapters_loaded": false,
  "weights_updated": false,
  "dataset_accepted": false,
  "modes": {
    "answer_only": {
      "answer_scoring": {
        "version": "answer-only-v2",
        "answer_matches": 9,
        "format_violations": 809,
        "token_limit_hits": 0,
        "non_year_answer_matches": 0,
        "year_answer_matc

In [5]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['training_updates'] == 0 and not summary['dataset_accepted']
subprocess.run([sys.executable, '-m', 'src.evaluation.corrected_m0', '--config', config_path, '--output-dir', str(output_dir), '--mode', 'verify'], check=True)
if summary['execution_complete']:
    print('Replay controls passed. New GPU responses:', summary['new_predictions'], '| reused parent responses:', summary['reused_predictions'])
    for mode, data in summary['modes'].items():
        score = data['answer_scoring']
        print(mode, '| matches:', score['answer_matches'], '/1260', '| non-year matches:', score['non_year_answer_matches'], '| format:', score['format_violations'], '| caps:', score['token_limit_hits'])
    print('Download evidence for corrected-row review before dataset acceptance/full M1 preparation.')
else:
    print('Controls did not reproduce. No baseline merged. Download the failure evidence; do not retry selectively.')


{
  "verified": true,
  "execution_complete": true,
  "replay_controls_passed": true,
  "new_records_checked": 80,
  "derived_records_reconstructed": 2520,
  "reused_records_checked_against_parent": 2504,
  "dataset_accepted": false,
  "note": "Score/token-metadata verification, not model inference or tokenizer-decode replay."
}
Replay controls passed. New GPU responses: 80 | reused parent responses: 2504
answer_only | matches: 9 /1260 | non-year matches: 0 | format: 809 | caps: 0
thinking | matches: 1 /1260 | non-year matches: 0 | format: 751 | caps: 16
Download evidence for corrected-row review before dataset acceptance/full M1 preparation.


In [6]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)


Download from Kaggle Output: /kaggle/working/m0-qwen17-corrected-aqe8clfe-evidence.zip
